# Introduction to Building AI Agents with LangChain

An **AI Agent** combines a Large Language Model (LLM) with **Tools** (Python functions) inside a reasoning loop. Instead of just generating text, an agent can decide *which* Python function to call, inspect the output, and iterate until it solves the user's request.

### How All Your Previous Python Concepts Connect Here!
* **Functions & Type Hints:** Every agent tool is a regular Python function decorated with `@tool`.
* **Docstrings:** The LLM reads your function's docstring to understand *when* and *how* to use the tool.
* **Dictionaries & Lists:** Used as mock databases/inventories and for passing conversation `messages` into the agent.
* **Pydantic (`BaseModel`, `Literal`, `@field_validator`):** Used to validate structured tool inputs and enforce strict output formats (`response_format`).
* **Loops:** Behind the scenes, `create_agent` runs a `while`-style loop: **LLM thinks $
ightarrow$ calls Python tool $
ightarrow$ reads result $
ightarrow$ answers user**.

### Setup & Installation
Run the cell below once if you haven't installed LangChain yet.

---
## Example 1: Minimal "Hello World" Agent (Weather Lookup)
In this first example, we define a simple Python function with `@tool`, pass it to `create_agent`, and let the LLM decide when to invoke it.

In [3]:
import os
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.tools import tool
from openai import OpenAI

# 1. Load environment variables from your .env file
# (Note: standard filename is ".env", not ".evn" — adjust the path if your file is elsewhere)
env_path = "/Users/nareshchaurasia/nc/PYTHON-ARCHITECT/Python-Immersive-AI-MAC/.env"
loaded = load_dotenv(dotenv_path=env_path, override=True)

# Optional fallback if running from a subdirectory and not using the hardcoded path above:
# if not loaded:
#     load_dotenv(dotenv_path=os.path.join("..", ".env"), override=True)

# 2. Verify OPENAI_API_KEY was loaded
assert os.environ.get("OPENAI_API_KEY"), (
    f"OPENAI_API_KEY not found. Check that it is set in {env_path}"
)

# The OpenAI client reads OPENAI_API_KEY from the environment automatically
client = OpenAI()

# Use a valid OpenAI model identifier (e.g., "gpt-4o-mini", "gpt-4.1-mini", or whichever model your account supports)
MODEL = "gpt-4o-mini"
print(f"Client ready. Using model: {MODEL}")

Client ready. Using model: gpt-4o-mini


In [ ]:
# 1. Define a Python function as a Tool using the @tool decorator
# NOTE: Type hints (city: str) and the docstring are CRITICAL—the LLM reads them!
@tool
def get_weather(city: str) -> str:
    """Returns the current weather forecast for a given city."""
    # Simulated weather lookup dictionary
    weather_data = {
        "pune": "28°C, Partly Cloudy",
        "london": "14°C, Light Rain",
        "tokyo": "21°C, Clear Skies"
    }
    return weather_data.get(city.lower(), f"Weather data unavailable for {city}.")

# 2. Create the Agent by combining an LLM model, tools list, and a system prompt
# You can swap "openai:gpt-4o-mini" with "google_genai:gemini-2.5-flash" or "ollama:llama3.2"
agent = create_agent(
    model="openai:gpt-4o-mini",
    tools=[get_weather],
    system_prompt="You are a concise, helpful assistant. Always use tools for weather facts."
)

# 3. Invoke the agent with a user message
response = agent.invoke({
    "messages": [{"role": "user", "content": "What is the weather like in Pune right now?"}]
})

# 4. Print the step-by-step message trace (Human -> AI Tool Call -> Tool Output -> Final AI Answer)
for msg in response["messages"]:
    print(f"[{msg.type.upper()}]: {msg.content or msg.tool_calls}")

Failed to multipart ingest runs: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


[HUMAN]: What is the weather like in Pune right now?
[AI]: [{'name': 'get_weather', 'args': {'city': 'Pune'}, 'id': 'call_Kh32JpmTLrPdOnKHAAN7qC5J', 'type': 'tool_call'}]
[TOOL]: 28°C, Partly Cloudy
[AI]: The current weather in Pune is 28°C and partly cloudy.


Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')
Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


---
## Example 2: Real-World Use Case — Customer Support & Inventory Agent
This use case combines **Dictionaries**, **Pydantic validation** (`BaseModel`, `Literal`, `@field_validator`), **Exceptions** (`try/except`), and **Multiple Tools** into a practical e-commerce support assistant.

In [ ]:
from typing import Literal
from pydantic import BaseModel, Field, field_validator
from langchain.agents import create_agent
from langchain.tools import tool

# --- Simulated Store Databases (Dictionaries) ---
ORDERS_DB = {
    "ORD-101": {"item": "Mechanical Keyboard", "status": "Shipped", "eta": "2 days"},
    "ORD-102": {"item": "USB-C Hub", "status": "Processing", "eta": "4 days"},
}

INVENTORY_DB = {
    "keyboard": {"price": 79.99, "stock": 12},
    "mouse": {"price": 29.99, "stock": 0},
    "monitor": {"price": 249.99, "stock": 5},
}

# --- Pydantic Schema for Validating Return Requests ---
class ReturnRequestInput(BaseModel):
    order_id: str = Field(description="Order ID in format ORD-XXX, e.g., ORD-101")
    reason: Literal["damaged", "wrong_item", "no_longer_needed"] = Field(
        description="Reason for returning the product"
    )

    @field_validator("order_id")
    @classmethod
    def validate_order_id(cls, v: str) -> str:
        cleaned = v.strip().upper()
        if not cleaned.startswith("ORD-"):
            raise ValueError("order_id must start with 'ORD-'")
        return cleaned

# --- Tool 1: Check Order Status ---
@tool
def check_order_status(order_id: str) -> str:
    """Looks up the shipping status and ETA of a customer order by order_id (e.g., ORD-101)."""
    order = ORDERS_DB.get(order_id.strip().upper())
    if not order:
        return f"Order '{order_id}' was not found in our system."
    return f"Order {order_id.upper()} ({order['item']}): Status is '{order['status']}', ETA: {order['eta']}."

# --- Tool 2: Check Product Stock & Price ---
@tool
def check_product_stock(product_name: str) -> str:
    """Checks the price and available stock quantity for a product."""
    product = INVENTORY_DB.get(product_name.strip().lower())
    if not product:
        return f"Product '{product_name}' is not carried in our catalog."
    if product["stock"] == 0:
        return f"'{product_name}' is currently OUT OF STOCK (Price: ${product['price']})."
    return f"'{product_name}' is IN STOCK ({product['stock']} units available) at ${product['price']}."

# --- Tool 3: Submit a Return Request (using Pydantic args_schema) ---
@tool(args_schema=ReturnRequestInput)
def submit_return_request(order_id: str, reason: str) -> str:
    """Submits a product return request after validating the order ID and return reason."""
    if order_id not in ORDERS_DB:
        return f"Cannot process return: Order {order_id} does not exist."
    return f"Return approved for {order_id} (Reason: {reason}). A prepaid return label has been emailed."

# --- Build the Support Agent ---
support_agent = create_agent(
    model="openai:gpt-4o-mini",
    tools=[check_order_status, check_product_stock, submit_return_request],
    system_prompt=(
        "You are a polite customer support agent for TechStore. "
        "Use the provided tools to check orders, verify stock, or process returns. "
        "Do not guess order statuses or stock numbers."
    )
)

# --- Test the Agent on a Multi-Step Customer Query ---
user_query = (
    "Hi! Can you check the status of order ORD-101? "
    "Also, is the mouse in stock, and can I return order ORD-102 because I received the wrong item?"
)

result = support_agent.invoke({
    "messages": [{"role": "user", "content": user_query}]
})

# Display the final response from the agent
print("=== FINAL AGENT RESPONSE ===")
print(result["messages"][-1].content)

---
## Example 3: Enforcing Structured Output with Pydantic (`response_format`)
Sometimes you want an agent not only to use tools, but also to return its final answer as a **validated Pydantic object** so downstream Python code can reliably read fields without parsing messy text.

In [ ]:
from typing import Literal
from pydantic import BaseModel, Field
from langchain.agents import create_agent
from langchain.tools import tool

# 1. Define the desired structured output schema using Pydantic
class TicketSummary(BaseModel):
    customer_intent: Literal["order_tracking", "stock_inquiry", "return_request", "general"]
    urgency: Literal["low", "medium", "high"]
    action_taken: str = Field(description="Short summary of what the agent did")
    resolved: bool = Field(description="True if the customer's issue was resolved")

# 2. Reuse our check_order_status tool from Example 2
@tool
def lookup_ticket_order(order_id: str) -> str:
    """Fetches order details by ID."""
    sample_orders = {"ORD-500": "Delayed due to weather; new ETA tomorrow."}
    return sample_orders.get(order_id.upper(), "Order not found.")

# 3. Pass response_format=TicketSummary into create_agent
triage_agent = create_agent(
    model="openai:gpt-4o-mini",
    tools=[lookup_ticket_order],
    system_prompt="You triage support tickets, check order details if needed, and output structured metadata.",
    response_format=TicketSummary
)

# 4. Run the agent and access the structured Pydantic instance
response = triage_agent.invoke({
    "messages": [{"role": "user", "content": "Where is my urgent order ORD-500?!"}]
})

structured_output = response["structured_response"]
print("Structured Object:", structured_output)
print("Intent:", structured_output.customer_intent)
print("Urgency:", structured_output.urgency)
print("Resolved:", structured_output.resolved)

---
## Key Takeaways
1. **Agents = LLM + Tools + Loop:** `create_agent(model, tools, system_prompt)` automatically manages the loop of calling tools and feeding results back to the LLM.
2. **Python Functions Become Tools:** Decorate regular Python functions with `@tool`. Clear **type hints** and **docstrings** act as the instructions telling the LLM when and how to call each tool.
3. **Pydantic Integrates Naturally:** Use Pydantic `BaseModel` + `Literal` + `@field_validator` both to validate tool inputs (`args_schema`) and to guarantee structured final responses (`response_format`).